# 🧱 Part 20: Vision-Language Models: Connecting Images to Text

> **Previous context**: Text-only models read token sequences. VLMs add visual information to the same language-model pipeline.
> **Goal for this part**: Understand patch embeddings, vision encoders, projectors, Cross-Attention, and Flamingo-style gated fusion.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. Why VLMs need a bridge

Images are arrays of pixels; language models expect token-like vectors. A VLM must convert visual signals into representations the LLM can use.

## 1. Patch Embedding

An image can be split into patches, and each patch can become a vector, similar to a visual token.

## 2. Projector and fusion

A projector maps vision features into the language model dimension. Cross-Attention or gated blocks let text tokens read image features.

## 3. Training strategy

Many VLMs freeze some parts at first so visual ability grows without immediately destroying language ability.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] Images become patch or vision features.
- [ ] A projector aligns visual features with the LLM hidden size.
- [ ] Cross-Attention lets language tokens read visual information.

Next, continue through the code cells for the Frontiers part and inspect the printed observations.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

torch.manual_seed(42)

In [ ]:
# Teaching note: follow this line to see the main step.
IMG_SIZE = 224
PATCH_SIZE = 16

# Teaching note: follow this line to see the main step.
fake_image = torch.randn(3, IMG_SIZE, IMG_SIZE)

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
patches = fake_image.unfold(1, PATCH_SIZE, PATCH_SIZE).unfold(2, PATCH_SIZE, PATCH_SIZE)
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

# Teaching note: follow this line to see the main step.
num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
patch_dim = 3 * PATCH_SIZE * PATCH_SIZE  # 3×16×16 = 768
patches_flat = patches.permute(1, 2, 0, 3, 4).reshape(num_patches, patch_dim)

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
class PatchEmbedding(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, img_size=224, patch_size=16, in_channels=3, d_model=512):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        
        # Teaching note: follow this line to see the main step.
        # Teaching note: follow this line to see the main step.
        self.proj = nn.Conv2d(in_channels, d_model, 
                              kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        '"""\n        Input: x shape = [batch, 3, 224, 224]\n        Output:   shape = [batch, num_patches, d_model]\n        """'
        x = self.proj(x)          # [batch, d_model, 14, 14]
        x = x.flatten(2)           # [batch, d_model, 196]
        x = x.transpose(1, 2)      # [batch, 196, d_model]
        return x

# Teaching note: follow this line to see the main step.
patch_emb = PatchEmbedding(img_size=224, patch_size=16, d_model=512)
dummy_img = torch.randn(2, 3, 224, 224)  # Teaching note: follow this line to see the main step.
visual_tokens = patch_emb(dummy_img)

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
d_model = 512
text_vocab_size = 1000

# Teaching note: follow this line to see the main step.
text_ids = torch.tensor([[5, 12, 78, 3, 90]])  # Teaching note: follow this line to see the main step.
text_emb = nn.Embedding(text_vocab_size, d_model)
text_vecs = text_emb(text_ids)  # [1, 5, 512]

# Teaching note: follow this line to see the main step.
visual_vecs = torch.randn(1, 196, d_model)  # Teaching note: follow this line to see the main step.

# Teaching note: follow this line to see the main step.
combined = torch.cat([visual_vecs, text_vecs], dim=1)

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
class MiniVLM(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, text_vocab_size=100, d_model=128, num_heads=2, num_layers=2):
        super().__init__()
        self.vision = PatchEmbedding(img_size=224, patch_size=16, in_channels=3, d_model=d_model)
        self.text_embedding = nn.Embedding(text_vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=4 * d_model,
            batch_first=True,
        )
        self.layers = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, text_vocab_size)

    def forward(self, images, text_ids):
        visual_tokens = self.vision(images)
        text_tokens = self.text_embedding(text_ids)
        x = torch.cat([visual_tokens, text_tokens], dim=1)
        x = self.layers(x)
        x = self.norm(x)
        return self.lm_head(x)

# Teaching note: follow this line to see the main step.
vlm = MiniVLM(text_vocab_size=100, d_model=128, num_heads=2, num_layers=2)

# Teaching note: follow this line to see the main step.
dummy_images = torch.randn(1, 3, 224, 224)
dummy_text = torch.randint(0, 100, (1, 10))  # Teaching note: follow this line to see the main step.

logits = vlm(dummy_images, dummy_text)

print('Read the values printed above and connect them to the concept in this cell.')
print('f"Output: {logits.shape}"')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')